# ARIMA

Owner: _(name)_

Rules shared by all three model notebooks:

- read the data and the split from `src` (never re-download, never re-split);
- fit on train, choose settings on validation, touch test once at the end;
- save test predictions with `common.save_predictions("arima", test, predictions)`.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/OWNER/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
df.tail()

## 1. Train / validation / test

In [ ]:
y = common.target_series(df)
train, val, test = common.split(y)
print(len(train), len(val), len(test))

## 2. Choose p, d, q

`auto_arima` runs the stepwise search from the lecture (KPSS for d, then AICc for p and q).
On the log return expect d = 0 and small p, q; on the price expect d = 1.

In [ ]:
import pmdarima as pm
auto = pm.auto_arima(train, seasonal=False, stepwise=True, suppress_warnings=True, information_criterion="aicc", trace=True)
print(auto.summary())

## 3. One-step-ahead forecasts on validation, then test

Each day the model forecasts the next value, then is told the true value (`update`), like a real
forecaster. This is slow but honest.

In [ ]:
def walk_forward(model, series):
    preds = []
    for t, value in series.items():
        preds.append(float(np.asarray(model.predict(n_periods=1))[0]))
        model.update([value])
    return pd.Series(preds, index=series.index)

model = auto
val_pred = walk_forward(model, val)
print("validation MAE", round(common.mae(val, val_pred), 6), "RMSE", round(common.rmse(val, val_pred), 6))

In [ ]:
test_pred = walk_forward(model, test)      # the model has now seen train + validation
print("test MAE", round(common.mae(test, test_pred), 6), "RMSE", round(common.rmse(test, test_pred), 6))
common.save_predictions("arima", test, test_pred)

## 4. Residual check (the workflow's step 4)

In [ ]:
resid = (test - test_pred).dropna()
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
resid.plot(ax=ax[0], title="residuals over time")
plot_acf(resid, lags=30, ax=ax[1], title="ACF of residuals (should sit inside the limits)")
plt.tight_layout()